# Fine-tuning a Frontier Model

We will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [1]:
# Imports:

import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

## Loading in Dataset:

In [2]:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

train, val, test = Item.from_hub(dataset_name= 'ed-donner/items_lite')

print(f'Loaded {len(train)} train items, {len(val)} validation items, {len(test)} test items')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 20000 train items, 1000 validation items, 1000 test items


## Data Size:

OpenAI recommends fine-tuning with a small population of 50-100 examples



Fine-Tuning with full 20,000 points will cost more than 3$. - We will use 1000 data points to train and 100 for validation.

In [3]:
fine_tune_train = train[:1000]
fine_tune_validation = val[:100]

In [4]:
print(len(fine_tune_train), len(fine_tune_validation))

1000 100


## Step 1:

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI.

In [12]:
openai = OpenAI()

In [5]:
def messages_for(item):
    message = f'Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}'

    return [
        {'role': 'user', 'content': message},
        {'role': 'assistant', 'content': f'${item.price:.2f}'}
    ]

In [6]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [7]:
# Converting the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices.

def make_jsonl(items):
    result = ""

    for item in items:
        messages = messages_for(item)
        messages_str =  json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'

    return result.strip()

In [8]:
print(make_jsonl(fine_tune_train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [9]:
# Converting the items into jsonl and writing them to a file.

def write_jsonl(items, filename):
    with open(filename, 'w') as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [10]:
# JSONL File for Training:
write_jsonl(items= fine_tune_train, filename= 'jsonl/fine_tune_train.jsonl')

In [11]:
# JSONL File for Validation:
write_jsonl(items= fine_tune_validation, filename= 'jsonl/fine_tune_validation.json')

In [13]:
# Creating File-Tune Training and Validation Files in OpenAI:

with open('jsonl/fine_tune_train.jsonl', 'rb') as f:
    train_file = openai.files.create(file= f, purpose= 'fine-tune')

with open('jsonl/fine_tune_validation.json', 'rb') as f:
    validation_file = openai.files.create(file= f, purpose= 'fine-tune')

In [16]:
train_file

FileObject(id='file-BCdQMgZC6EBhWcZZ1ZAS2d', bytes=551342, created_at=1778071459, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [15]:
validation_file

FileObject(id='file-TBV2pwBqrwfnSFDfBJSSFJ', bytes=55911, created_at=1778071461, filename='fine_tune_validation.json', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

 We can See the files from here:
 https://platform.openai.com/storage/files

## Step 2:

Fine-Tuning

In [17]:
# Creating Fine-Tuning Job:
openai.fine_tuning.jobs.create(
    training_file= train_file.id,
    validation_file= validation_file.id,
    model= 'gpt-4.1-nano-2025-04-14',
    seed= 42,
    hyperparameters= {'n_epochs': 2, 'batch_size': 16},
    suffix= 'pricer'
)

FineTuningJob(id='ftjob-meAAY5tjO6EQmLKNa0HnMbQ5', created_at=1778072063, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=2), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-KcikX2bGsCDOknIfs2t9IKtw', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-BCdQMgZC6EBhWcZZ1ZAS2d', validation_file='file-TBV2pwBqrwfnSFDfBJSSFJ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=2))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [18]:
openai.fine_tuning.jobs.list()

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-meAAY5tjO6EQmLKNa0HnMbQ5', created_at=1778072063, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-KcikX2bGsCDOknIfs2t9IKtw', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-BCdQMgZC6EBhWcZZ1ZAS2d', validation_file='file-TBV2pwBqrwfnSFDfBJSSFJ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_

In [19]:
job_id = openai.fine_tuning.jobs.list(limit= 1).data[0].id

In [20]:
job_id

'ftjob-meAAY5tjO6EQmLKNa0HnMbQ5'

In [21]:
openai.fine_tuning.jobs.retrieve(fine_tuning_job_id= job_id)

FineTuningJob(id='ftjob-meAAY5tjO6EQmLKNa0HnMbQ5', created_at=1778072063, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-KcikX2bGsCDOknIfs2t9IKtw', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-BCdQMgZC6EBhWcZZ1ZAS2d', validation_file='file-TBV2pwBqrwfnSFDfBJSSFJ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [31]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id= job_id, limit= 3).data

[FineTuningJobEvent(id='ftevent-kkwMvsn1TLdLmjV2gyvpUKU7', created_at=1778073309, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-nT2Tdomr1RbaEnhi7AAtw0cx', created_at=1778073306, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-wLMQ6Pje0oSmtMHzML0WvDED', created_at=1778073306, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:pricer:DcWBu0np passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking

View Fine-Tuning Process here:
https://platform.openai.com/finetune/

## Step 3:

Testing Fine-Tuned Model

In [32]:
# Retrieving fine-tuned model's name:

openai.fine_tuning.jobs.retrieve(fine_tuning_job_id= job_id)

FineTuningJob(id='ftjob-meAAY5tjO6EQmLKNa0HnMbQ5', created_at=1778072063, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:personal:pricer:DcWBu0np', finished_at=1778072575, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-KcikX2bGsCDOknIfs2t9IKtw', result_files=['file-P7aC9ZTEwyQEFez8diww5f'], seed=42, status='succeeded', trained_tokens=225424, training_file='file-BCdQMgZC6EBhWcZZ1ZAS2d', validation_file='file-TBV2pwBqrwfnSFDfBJSSFJ', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=2))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_exec

In [33]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(fine_tuning_job_id= job_id).fine_tuned_model

In [34]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:DcWBu0np'

In [35]:
# Prompt for testing:

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {'role': 'user', 'content': message},
    ]

In [36]:
test_messages_for(fine_tune_validation[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: SAFUEL 10,000mAh Magnetic Portable Charger  \nCategory: Electronics  \nBrand: SAFUEL  \nDescription: A compact 10,000mAh power bank that snaps magnetically to MagSafe‑enabled iPhones for fast wireless charging.  \nDetails: Offers 20W USB‑C PD, QC 3.0 wired output, auto shut‑off, and a built‑in LED indicator.'}]